# W12C2 Lab: Chain-of-Thought and Self-Consistency

Run every cell from the top. **Everything already works.**

Needs the local model running; falls back to placeholders if not.

Today you will:

1. Watch a small model fail arithmetic, then fix it with one sentence.
2. Sample the same question many times and take a vote.
3. Find where chain-of-thought does NOT help.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

PROBLEMS = [
    ("A shop sells pens at 3 dollars each. Ana buys 4 pens and pays with a 20 dollar note. How much change?", 8),
    ("Tom has 15 apples. He gives 4 to Sam and 3 to Mia. How many are left?", 8),
    ("A train travels 60 km in 1 hour. How far in 3 hours?", 180),
    ("There are 24 students. One third leave. How many remain?", 16),
    ("A book costs 12 dollars. You buy 3 and get 5 dollars off. What do you pay?", 31),
]
print(f"{len(PROBLEMS)} word problems with known answers")

In [ ]:
# GIVEN. One helper for talking to the local model.
import ollama

MODEL = "qwen2.5:0.5b"
_offline_notice_shown = False

def ask(prompt, temperature=0.0, n=1):
    """Send a prompt to the local model. Returns a list of n replies.

    If Ollama is not running you get a fixed placeholder instead, so the
    notebook still executes end to end. Start it with:
        docker compose -f docker/docker-compose.yml up -d ollama
    """
    global _offline_notice_shown
    out = []
    for _ in range(n):
        try:
            r = ollama.chat(model=MODEL,
                            messages=[{"role": "user", "content": prompt}],
                            options={"temperature": temperature})
            out.append(r["message"]["content"].strip())
        except Exception:
            if not _offline_notice_shown:
                print("[no local model running: using placeholder replies]")
                _offline_notice_shown = True
            out.append("(placeholder)")
    return out

print("model:", MODEL)
print("test :", ask("Reply with exactly the word: ready")[0][:40])

## Part 1. Ask for the answer, and ask for the working

Same questions, two prompts. The only difference is one sentence telling
the model to show its steps.

In [ ]:
# GIVEN. Pull the final number out of a reply, then score both prompts.
def final_number(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(numbers[-1]) if numbers else None

DIRECT = "{q}\n\nAnswer with the number only."
COT = "{q}\n\nThink step by step, then give the final number on its own line."

def run(template, temperature=0.0):
    rows = []
    for q, gold in PROBLEMS:
        reply = ask(template.format(q=q), temperature=temperature)[0]
        got = final_number(reply)
        rows.append({"gold": gold, "got": got, "right": got == gold,
                     "reply": reply.replace("\n", " ")[:44]})
    return pd.DataFrame(rows)

direct = run(DIRECT)
cot = run(COT)
print("--- direct ---");  print(direct.to_string(index=False))
print()
print("--- chain of thought ---"); print(cot.to_string(index=False))
print()
print(f"direct accuracy          : {direct['right'].mean():.2f}")
print(f"chain-of-thought accuracy: {cot['right'].mean():.2f}")

In [ ]:
# ================== YOUR TURN 1 ==================
# Write your own step-by-step instruction and see whether the exact
# wording matters.
#
# Try: 'Work through it carefully one step at a time.'
# Then try: 'Explain your reasoning before answering.'
#
# Expected: different phrasings give different accuracy, sometimes by a lot, on a
#           model this small. That instability is the honest state of prompt
#           engineering: it works, and nobody can tell you in advance which words
#           will work.
# ===============================================
MY_COT = "{q}\n\nThink step by step, then give the final number on its own line."   # <-- change

mine = run(MY_COT)
print(mine.to_string(index=False))
print(f"\naccuracy: {mine['right'].mean():.2f}")

## Part 2. Self-consistency: ask more than once

A single sample can go wrong anywhere. Sample the same question several
times at a nonzero temperature and take the commonest answer.

In [ ]:
# GIVEN. Majority vote over several samples.
def self_consistent(q, n=5, temperature=0.8):
    replies = ask(COT.format(q=q), temperature=temperature, n=n)
    answers = [final_number(r) for r in replies]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None, []
    return Counter(answers).most_common(1)[0][0], answers

rows = []
for q, gold in PROBLEMS:
    vote, samples = self_consistent(q, n=5)
    rows.append({"gold": gold, "vote": vote, "right": vote == gold,
                 "samples": str(samples)[:34]})
sc = pd.DataFrame(rows)
print(sc.to_string(index=False))
print(f"\nself-consistency accuracy: {sc['right'].mean():.2f}")

In [ ]:
# ================== YOUR TURN 2 ==================
# How many samples are worth paying for? Try N = 1, then 3, then 9.
#
# Every sample is another full call, so this is a direct cost decision.
#
# Expected: accuracy usually rises then flattens while cost rises linearly forever.
#           With five problems the numbers are noisy, so re-run before believing a
#           difference. Voting only helps when the model is right MORE often than
#           wrong: it amplifies whatever the model already tends to do.
# ===============================================
N = 3          # <-- try 1, then 9

rows = []
for q, gold in PROBLEMS:
    vote, samples = self_consistent(q, n=N)
    rows.append(vote == gold)

print(f"N = {N}   accuracy {sum(rows) / len(rows):.2f}   "
      f"model calls used: {N * len(PROBLEMS)}")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Wording changes the score, and not predictably. On a 0.5B model the variance
#   between phrasings can be as large as the effect of chain-of-thought itself,
#   which is exactly why the Wei 2022 paper reports results averaged over
#   several prompts rather than a single lucky one.
#
# YOUR TURN 2
#   Accuracy climbs from 1 to about 3 or 5 samples and then flattens, while cost
#   grows linearly with no ceiling. Self-consistency is a majority vote, so it
#   only helps when correct answers are already the most common single outcome.
#   If the model is reliably wrong in the same way, voting makes it worse.